In [8]:
import os
import torch
import ast
import re
import warnings
import hashlib

import pandas as pd
import numpy as np
import networkx as nx

from collections import defaultdict
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Embedding using: {device}")

# Move Model to GPU once
tokenizer = AutoTokenizer.from_pretrained("jjzha/dajobbert-base-uncased")
embed_model = AutoModel.from_pretrained("jjzha/dajobbert-base-uncased").to(device)
embed_model.eval()

Embedding using: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31748, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [10]:
def deep_clean(name):
    name = str(name).lower()
    # Safely strip any standard RDF/OWL namespace prefix
    name = re.sub(r'^[a-z]+:', '', name) 
    name = re.sub(r'xsd:|_jie_|_jip_', '', name)
    
    # Strip trailing numbers (e.g., position_1527549 -> position)
    name = re.sub(r'_\d+$', '', name)
    
    # Strip long hex strings (e.g., candidate_1ea333bcf4... -> candidate)
    name = re.sub(r'_[a-f0-9]{8,}$', '', name) 
    
    return name.strip('_').replace('_', ' ').strip()

def get_ntype(node_str):
    raw = str(node_str).upper()
    if "CANDIDATE" in raw: return "candidate"
    if "POSITION" in raw or "VACANCY" in raw: return "vacancy"
    return "attribute"

def build_embedding_cache(hits_misses, batch_size=64):
    unique_names = set()
    for graphs in hits_misses.values():
        for g in graphs.values():
            for node in g.nodes():
                c_n = deep_clean(node)
                if not (c_n.isdigit() or c_n in ['decimal', 'integer', 'string', '']):
                    unique_names.add(c_n)
    
    name_list = list(unique_names)
    cache = {}
    
    for i in range(0, len(name_list), batch_size):
        batch_names = name_list[i : i + batch_size]
        inputs = tokenizer(batch_names, return_tensors="pt", padding=True, 
                           truncation=True, max_length=64).to(device)
        with torch.no_grad():
            outputs = embed_model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu()
        for name, emb in zip(batch_names, embeddings):
            cache[name] = emb
            
    return cache

In [11]:
def get_split(user_id, train_size=0.8, val_size=0.05):
    hash_val = int(hashlib.md5(str(user_id).encode('utf-8')).hexdigest(), 16)
    bucket = (hash_val % 1000) / 1000.0  
    if bucket < train_size: return "train"
    elif bucket < (train_size + val_size): return "val"
    else: return "test"

def create_dataloaders(hits_misses, truth_dict, embedding_cache, train_size=0.8, val_size=0.1):
    embedding_dim = embed_model.config.hidden_size
    train_list, val_list, test_list = [], [], []
    u_keys = sorted(list(hits_misses.keys())) 

    test_dict = defaultdict(list)

    for i, user_id in tqdm(enumerate(u_keys), total=len(u_keys)):
        split = get_split(user_id, train_size, val_size)
        target_list = train_list if split == "train" else (val_list if split == "val" else test_list)            
            
        graphs = hits_misses[user_id]
        temp_graphs, truth_values = [], []
        raw_tail_ids = []
        
        node_info = {}
        global_node_counter = 1
        
        target_cand_names = []
        target_vac_names = []

        if split == "test":
            test_dict[user_id].extend([tail_id for tail_id, _ in graphs.items()])

        for sg_idx, (tail_id, G_raw) in enumerate(graphs.items()):
            if not G_raw: continue
            
            safe_tail_id = str(tail_id).split('.')[0] 
            
            cand_node, vac_node = None, None
            for n in list(G_raw.nodes()):
                n_str = str(n).lower()
                if str(user_id).lower() in n_str and "candidate" in n_str: cand_node = n
                if safe_tail_id in n_str and ("position" in n_str or "vacancy" in n_str): vac_node = n

            # Add Artificial Edge
            if cand_node and vac_node:
                G_raw.add_edge(cand_node, vac_node, edge_type="gets_recommended")
                undirected = G_raw.to_undirected()
                if nx.has_path(undirected, cand_node, vac_node):
                    main_comp = nx.node_connected_component(undirected, cand_node)
                    G_raw = G_raw.subgraph(main_comp).copy()
                else: continue
            else:
                continue 

            valid_nodes_in_sg = []
            sg_cand_new_name, sg_vac_new_name = None, None
            
            for n in G_raw.nodes():
                c_n = deep_clean(n)
                if c_n in embedding_cache:
                    ntype = get_ntype(n)
                    new_name = f"{n}_{i}_{sg_idx}"
                    node_info[new_name] = (ntype, global_node_counter, sg_idx)
                    G_raw.nodes[n]['x'] = embedding_cache[c_n]
                    
                    if n == cand_node: sg_cand_new_name = new_name
                    if n == vac_node: sg_vac_new_name = new_name
                    
                    valid_nodes_in_sg.append((n, new_name))
                    global_node_counter += 1

            if valid_nodes_in_sg and sg_cand_new_name and sg_vac_new_name:
                mapping = {old: new for old, new in valid_nodes_in_sg}
                sg_final = G_raw.subgraph([n for n, _ in valid_nodes_in_sg])
                temp_graphs.append(nx.relabel_nodes(sg_final, mapping))
                
                target_cand_names.append(sg_cand_new_name)
                target_vac_names.append(sg_vac_new_name)
                
                u_truth = truth_dict.get(user_id, truth_dict.get(str(user_id), {}))
                try: 
                    label = u_truth.get(tail_id, u_truth.get(int(float(tail_id)), 0))
                except:
                    label = 0
                truth_values.append(float(label))
                raw_tail_ids.append(str(tail_id))

        if not temp_graphs: continue

        merged_G = nx.compose_all(temp_graphs)
        data = HeteroData()
        node_map = {} 
        type_data = defaultdict(lambda: {"x":[], "u":[], "s":[]})
        
        for node in merged_G.nodes():
            ntype, uid, sg = node_info[node]
            node_map[node] = len(type_data[ntype]["x"])
            type_data[ntype]["x"].append(merged_G.nodes[node]['x'])
            type_data[ntype]["u"].append(uid)
            type_data[ntype]["s"].append(sg)

        for base_type in ["candidate", "vacancy", "attribute"]:
            if base_type in type_data:
                data[base_type].x = torch.stack(type_data[base_type]["x"])
                data[base_type].unique_node_id = torch.tensor(type_data[base_type]["u"])
                data[base_type].sub_graph = torch.tensor(type_data[base_type]["s"])
            else:
                data[base_type].x = torch.zeros((0, embedding_dim))
                data[base_type].unique_node_id = torch.empty((0,), dtype=torch.long)
                data[base_type].sub_graph = torch.empty((0,), dtype=torch.long)

        for u, v, d in merged_G.edges(data=True):
            u_t, _, _ = node_info[u]; v_t, _, _ = node_info[v]
            raw_etype = d.get("edge_type", d.get("type", d.get("label", d.get("predicate", "related_to"))))
            etype = deep_clean(raw_etype).replace(' ', '_')
            if not etype: etype = "related_to"
            
            triplet = (u_t, etype, v_t)
            if not hasattr(data[triplet], 'edge_index'):
                data[triplet].edge_index = torch.empty((2, 0), dtype=torch.long)
                
            new_e = torch.tensor([[node_map[u]], [node_map[v]]], dtype=torch.long)
            data[triplet].edge_index = torch.cat([data[triplet].edge_index, new_e], dim=1)

        for ntype in data.node_types:
            num_n = data[ntype].x.size(0)
            if num_n > 0:
                idx = torch.arange(num_n)
                data[ntype, "self_loop", ntype].edge_index = torch.stack([idx, idx], dim=0)

        # Map the saved anchor names to their proper PyTorch indices
        head_ids = [node_map[name] for name in target_cand_names]
        tail_ids = [node_map[name] for name in target_vac_names]

        data.y = torch.tensor(truth_values)
        data.head_nodes = torch.tensor(head_ids, dtype=torch.long)
        data.tail_nodes = torch.tensor(tail_ids, dtype=torch.long)

        data.cvid = [str(user_id)] * len(truth_values) 
        data.vacancy_id = raw_tail_ids
        
        target_list.append(data)
        
    return train_list, val_list, test_list, test_dict

In [13]:
os.makedirs("../dataloaders", exist_ok=True)

for isco in [True, False]:
    for model in ["qwen", "gemma", "llama"]:
        for prompt in ["structured", "semi-structured", "unstructured"]:
            
            # 1. Determine filenames
            suffix = "_isco" if isco else ""
            file_name = f"subgraphs_{model}_{prompt}{suffix}.xlsx"
            file_path = f"../kg_construction/{file_name}"
            
            if not os.path.exists(file_path):
                print(f"Skipping {file_name} (Not found)")
                continue
                
            print(f"\n{'='*50}\nProcessing: {file_name}\n{'='*50}")
            
            # 2. Load and filter Data
            df_graphs = pd.read_excel(file_path)
            df_graphs["response"] = df_graphs["response"].apply(lambda x: x if x == 0 else 1)
            
            non_zero = df_graphs.groupby("cvid")["response"].sum() > 0
            non_zero = set(non_zero[non_zero == True].index)
            df_graphs = df_graphs[df_graphs["cvid"].isin(non_zero)]
            
            # (Fixed pandas include_groups warning)
            truth_dict = df_graphs.groupby('cvid').apply(
                lambda x: dict(zip(x['vacancy'], x['response'])),
                include_groups=False 
            ).to_dict()

            # 3. Parse Subgraphs (Fixed positional warnings by using column names)
            hits = defaultdict(dict)
            misses = defaultdict(dict)
            
            for _, row in tqdm(df_graphs.iterrows(), total=len(df_graphs), desc=f"Parsing JSON ({model})"):
                if not pd.isna(row["graph"]):    
                    try:
                        data_dict = json.loads(row["graph"])
                    except:
                        try:
                            data_dict = ast.literal_eval(row["graph"])
                        except Exception as e:
                            print(e)
                            # The LLM cut off mid-generation or hallucinated bad syntax. 
                            continue
                    
                    g = nx.node_link_graph(data_dict)
                    
                    if row["response"] == 0:
                        misses[row["cvid"]][row["vacancy"]] = g
                    else:
                        hits[row["cvid"]][row["vacancy"]] = g

                else:
                    # If the LLM/extraction failed, we just get a graph with only the (candidate, vacancy) edge
                    g = nx.Graph()
                    g.add_edge(row["cvid"], int(row["vacancy"]))
                    
                    if row["response"] == 0:
                        misses[row["cvid"]][row["vacancy"]] = g
                    else:
                        hits[row["cvid"]][row["vacancy"]] = g
                        
            # Merge hits and misses for valid users
            hits_misses = {}
            valid_users = set(hits.keys()).intersection(set(misses.keys()))
            for user in valid_users:
                hits_misses[user] = {**hits[user], **misses[user]}

            if len(hits_misses) == 0:
                print(f"No valid hits/misses overlap for {file_name}. Skipping dataloader creation.")
                continue

            # 4. Embed and build DataLoaders
            print("Building cache...")
            cache = build_embedding_cache(hits_misses)
            
            print("Creating HeteroData...")
            train_loader, val_loader, test_loader, test_dict = create_dataloaders(hits_misses, truth_dict, cache)
            
            trainloader = DataLoader(train_loader) 
            valloader = DataLoader(val_loader) 
            testloader = DataLoader(test_loader)
            
            # 5. Save dynamically named DataLoaders
            train_path = f"../dataloaders/graph_trainloader_{model}_{prompt}{suffix}.pth"
            val_path = f"../dataloaders/graph_valloader_{model}_{prompt}{suffix}.pth"
            test_path = f"../dataloaders/graph_testloader_{model}_{prompt}{suffix}.pth"
            
            torch.save(trainloader, train_path)
            torch.save(valloader, val_path)
            torch.save(testloader, test_path)
            print(f"Saved dataloaders for {model}_{prompt}{suffix}!") 


Processing: subgraphs_qwen_structured_isco.xlsx


Parsing JSON (qwen):   0%|          | 0/7732 [00:00<?, ?it/s]

Building cache...
Creating HeteroData...


  0%|          | 0/367 [00:00<?, ?it/s]

Saved dataloaders for qwen_structured_isco!

Processing: subgraphs_qwen_semi-structured_isco.xlsx


...

Saved dataloaders for llama_semi-structured!

Processing: subgraphs_llama_unstructured.xlsx


Parsing JSON (llama):   0%|          | 0/7732 [00:00<?, ?it/s]

Building cache...
Creating HeteroData...


  0%|          | 0/367 [00:00<?, ?it/s]

Saved dataloaders for llama_unstructured!


In [ ]:
def check_dataloader_integrity(dataloader):
    # Grab the very first HeteroData graph from the loader
    data = next(iter(dataloader))
    
    
    print(f"\n1. Target Alignment (Head/Tail/Labels)")
    print(f"   - Number of Labels (y): {len(data.y)}")
    print(f"   - Number of Head Nodes (Candidates): {len(data.head_nodes)}")
    print(f"   - Number of Tail Nodes (Vacancies): {len(data.tail_nodes)}")
    
    if len(data.y) != len(data.head_nodes) or len(data.y) != len(data.tail_nodes):
        print("   ❌ ERROR: Mismatch between labels and target nodes!")
    else:
        print("   ✅ Labels and target nodes align perfectly.")

    print(f"\n2. The Target Edge ('gets_recommended')")
    target_edge = ('candidate', 'gets_recommended', 'vacancy')
    
    if target_edge in data.edge_types:
        edge_index = data[target_edge].edge_index
        num_edges = edge_index.shape[1]
        print(f"   - Found {num_edges} artificial target edges.")
        
        if num_edges > 0:
            print("\n   --- Edge to Label Mapping ---")
            cand_indices = edge_index[0].tolist()
            vac_indices = edge_index[1].tolist()
            labels = data.y.tolist()
            
            for i in range(min(5, num_edges)): # Print first 5 to check
                c_idx, v_idx, lbl = cand_indices[i], vac_indices[i], labels[i]
                print(f"     Edge {i+1}: Candidate[{c_idx}] -> Vacancy[{v_idx}] | Label = {lbl}")
    else:
        print("   ❌ CRITICAL ERROR: 'gets_recommended' edge is missing from schema!")

    print(f"\n3. Semantic Topology (Edge Distribution)")
    valid_bridges = 0
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.shape[1]
        # Ignore self-loops and the target edge for this count
        if num_edges > 0 and edge_type[1] != 'self_loop' and edge_type[1] != 'gets_recommended':
            print(f"   - {edge_type[0]} --[{edge_type[1]}]--> {edge_type[2]}: {num_edges} edges")
            valid_bridges += num_edges
            
    if valid_bridges == 0:
        print("   ❌ WARNING: No semantic bridges found! Graph might be disjoint.")
    else:
        print(f"   ✅ Found {valid_bridges} total semantic connections.")
        
    print("\n4. Tensor Shapes")
    print(f"   - Candidate Node Matrix (X): {data['candidate'].x.shape}")
    print(f"   - Vacancy Node Matrix (X): {data['vacancy'].x.shape}")
    print(f"   - Attribute Node Matrix (X): {data['attribute'].x.shape}")

# Run the test on one of your loaders
print("Testing Validation Loader...")
check_dataloader_integrity(valloader)